12. Implement batch gradient descent with early stopping for softmax
regression without using Scikit-Learn, only NumPy.

Using the iris data set

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# 1) Load Iris
iris = load_iris(as_frame=True)
X = iris.data.values                  # shape: (150, 4)
y = iris.target.values                # classes: 0, 1, 2
# 2) Train/validation/test split (for early stopping later)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)
# final ratios: train 60%, val 20%, test 20%
# 3) Scale features (important for gradient descent)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
# 4) Add bias column for NumPy implementation
X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_val_b = np.c_[np.ones((X_val.shape[0], 1)), X_val]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]
# 5) One-hot labels for softmax training
n_classes = len(np.unique(y))
Y_train = np.eye(n_classes)[y_train]
Y_val = np.eye(n_classes)[y_val]
Y_test = np.eye(n_classes)[y_test]
print("X_train_b:", X_train_b.shape, "Y_train:", Y_train.shape)
print("X_val_b:", X_val_b.shape, "Y_val:", Y_val.shape)
print("X_test_b:", X_test_b.shape, "Y_test:", Y_test.shape)

X_train_b: (90, 5) Y_train: (90, 3)
X_val_b: (30, 5) Y_val: (30, 3)
X_test_b: (30, 5) Y_test: (30, 3)


Building softmax regression from numpy

Buil 3 linear models for each class, logits are scores per class.

Softmax compares scores relative to each other and produces a prbability dist for scores.

One shared loss(cross entropy) is used to train all 3 models together.
Eg. Loss = - (0*log(p1) + 1*log(p2) + 0*log(p3)) = -log(p2)
Push true higher rest lower.

Early stopping: if the loss on validation starts increasing it means we are overfitting so we stop early.



In [4]:
import numpy as np
# -----------------------------
# 1) Initialize parameters
# -----------------------------
n_features = X_train_b.shape[1]   # includes bias column
n_classes = Y_train.shape[1]      # 3 classes in Iris -> setosa, versicolor, virginica
rng = np.random.default_rng(42)
Theta = rng.normal(0, 0.01, size=(n_features, n_classes))  # parameter matrix
# -----------------------------
# 2) Softmax helper
# -----------------------------
def softmax(logits):
    # logits shape: (m, n_classes)
    # subtract max for numerical stability
    z = logits - np.max(logits, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
# -----------------------------
# 3) Loss helper (cross-entropy)
# -----------------------------
def cross_entropy(y_true_onehot, y_proba):
    eps = 1e-12
    y_proba = np.clip(y_proba, eps, 1 - eps)
    return -np.mean(np.sum(y_true_onehot * np.log(y_proba), axis=1))
# -----------------------------
# 4) Batch Gradient Descent
# -----------------------------
eta = 0.1
n_epochs = 2000
for epoch in range(n_epochs):
    # Forward pass
    logits = X_train_b @ Theta                 # (m, 3)
    Y_hat = softmax(logits)                    # predicted probs
    # Gradient of softmax + cross-entropy
    m = X_train_b.shape[0]
    grad = (X_train_b.T @ (Y_hat - Y_train)) / m   # (n_features, 3)
    # Parameter update
    Theta -= eta * grad
    # Optional monitoring
    if epoch % 200 == 0:
        train_loss = cross_entropy(Y_train, Y_hat)
        print(f"epoch={epoch}, train_loss={train_loss:.4f}")
# -----------------------------
# 5) Validation / Test prediction
# -----------------------------
val_proba = softmax(X_val_b @ Theta)
val_pred = np.argmax(val_proba, axis=1)
val_acc = np.mean(val_pred == y_val)
test_proba = softmax(X_test_b @ Theta)
test_pred = np.argmax(test_proba, axis=1)
test_acc = np.mean(test_pred == y_test)
print(f"val_acc={val_acc:.4f}, test_acc={test_acc:.4f}")

epoch=0, train_loss=1.0965
epoch=200, train_loss=0.2333
epoch=400, train_loss=0.1653
epoch=600, train_loss=0.1323
epoch=800, train_loss=0.1128
epoch=1000, train_loss=0.1000
epoch=1200, train_loss=0.0908
epoch=1400, train_loss=0.0840
epoch=1600, train_loss=0.0787
epoch=1800, train_loss=0.0744
val_acc=0.9000, test_acc=0.9333
